# 01 · Exploring the synthetic customer book

What this notebook does, in order:
1. Generates a synthetic book with the project generator and checks it against the data contract.
2. Looks at each field's distribution and compares it with the profile in `docs/data_card.md`.
3. Shows how the model sees the book: risk buckets, late-payment rates and bucket transitions.
4. Solves the model once (no capital limit) and shows who gets offered an increase.

Everything here is synthetic. Run it from the repo root's environment (`uv run jupyter` or VS Code with the `.venv` kernel).

## 1. Generate and validate

`generate(n, seed)` draws each field independently (the reference book showed no correlation between fields) and validates the result against the pandera contract in `src/lending/contract.py`. Same `n` and `seed` always give the same table: that is what lets an Azure ML data asset be rebuilt exactly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lending import Book, Model, bucket, solve
from lending.synthetic import generate

df = generate(30_000, seed=42)
df.head()

`describe()` gives min / mean / max per field. Compare with the data card: loans 500–4,999, days 0–364, on-time 80–100%, increases 0 or 3–5.

In [ ]:
df.describe().round(2)

## 2. Distributions

One histogram per field. The first three should be flat (uniform); increases should show the gap at 1 and 2.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, col in zip(axes, ["loan_amount", "days_since_last_loan", "ontime_pct", "n_increases"], strict=True):
    ax.hist(df[col], bins=30 if col != "n_increases" else np.arange(8) - 0.5)
    ax.set_title(col)
plt.tight_layout()

## 3. How the model sees the book

**Risk buckets** come from the on-time rate: prime ≥ 95%, near-prime 85–95%, subprime < 85%. The mix should be about 25 / 50 / 25.

In [ ]:
names = ["prime", "near-prime", "subprime"]
r = bucket(df.ontime_pct.to_numpy() / 100)
pd.Series(np.bincount(r), index=names, name="customers").to_frame().assign(share=lambda t: (t.customers / len(df)).round(3))

`Book.from_frame` turns the table into arrays the solver uses (it validates again: the contract runs at every entry point). `Model.calibrate` then estimates, from the book:
- **late rates** per bucket (1 − average on-time rate), which drive default risk;
- **transition matrices**: the chance of moving between buckets in one cycle if we *hold* vs *grant* an increase. Granting shifts customers slightly towards worse buckets (leverage).

In [ ]:
book = Book.from_frame(df)
model = Model.calibrate(book)
print("late rate by bucket:", dict(zip(names, model.late.round(3).tolist(), strict=True)))
pd.concat(
    {"hold": pd.DataFrame(model.T_hold, names, names), "grant": pd.DataFrame(model.T_grant, names, names)}, axis=1
).round(3)

## 4. Solve once and look at the policy

`solve(book, model, lam=0.0)` runs backward induction for every customer with **no capital charge** (λ = 0). It returns a policy array indexed `[customer, bucket, increases so far, windows held]`. Below we read each customer's decision at their **first window** (k = 0, h = 0, their own bucket) and tabulate the share offered by bucket and loan size. Expect: prime and small loans offered most; large loans and riskier buckets held.

In [ ]:
pol, V = solve(book, model, lam=0.0)
first = pol[np.arange(len(book)), r, 0, 0]
bands = pd.cut(df.loan_amount, [0, 1500, 2500, 3500, 5000], labels=["<=1.5k", "1.5-2.5k", "2.5-3.5k", ">3.5k"])
pd.crosstab(pd.Series(pd.Categorical.from_codes(r, names), name="bucket"), bands, values=first, aggfunc="mean").round(2)

**What to take away:** the synthetic book reproduces the reference shape, the contract guards every entry point, and the unconstrained policy already concentrates offers on low-risk, small-loan customers. The capital limit (λ > 0, `find_lambda`) tightens this further; that is the nightly job's work.